# 2. Benchmark — easy-search + 评估

对每个方法跑 Foldseek **easy-search**（query = target），再按 scope_family 协议计算至第 1 个 wrong-fold FP 的灵敏度，写出 AUC CSV。

参数：`-s 9.5 --max-seqs 2000 -e 10`（与 `new_scope40` 一致）。

> easy-search 较吃 CPU。登录节点可把 `THREADS` 调小；完整跑建议在计算节点执行本 notebook。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "lib").is_dir():
    raise SystemExit(f"请在项目根目录启动 notebook，当前: {ROOT}")
sys.path.insert(0, str(ROOT))

from lib import config
from lib.search import search_all, search_one
from lib.evaluate import evaluate_all

SKIP_EXISTING = True
THREADS = config.EASY_SEARCH_PARAMS["threads"]  # 可改为 8 / 16
# 只跑部分方法时设为 key 列表，例如 ["foldseek", "ESM3_LoRA"]；None = 全部
ONLY_METHODS = None

print("方法:", [k for _, k, _ in config.METHODS])
print("THREADS =", THREADS)

## Phase A — easy-search

输出：`work/aln/{method}_easy.tsv`

In [ ]:
if ONLY_METHODS is None:
    aln_paths = search_all(skip_existing=SKIP_EXISTING, threads=THREADS)
else:
    aln_paths = {}
    for key in ONLY_METHODS:
        print(f"\n══ easy-search: {key} ══")
        aln_paths[key] = search_one(key, skip_existing=SKIP_EXISTING, threads=THREADS)

for key, path in aln_paths.items():
    nlines = sum(1 for _ in path.open()) if path.is_file() else 0
    print(f"{key:12s} → {path.name}  lines={nlines:,}")

## Phase B — 评估

1. 生成 `work/metrics/scop_lookup.tsv`
2. 写出 `*_fam/sup/fol.tsv`
3. 汇总 `work/metrics/auc_easy.csv`

In [ ]:
auc_df = evaluate_all(skip_existing=SKIP_EXISTING)
display(auc_df)
print("\nBenchmark 完成。下一步打开 3.plot.ipynb")